In [1]:
import torch
import torch.nn as nn


# =====================================================================
# 1. Vanilla Recurrent Neural Network (RNN)
# =====================================================================
class RNNClassifier(nn.Module):
    """
    Standard Vanilla RNN Classifier.
    Suffers from vanishing/exploding gradients on longer sequences.
    """

    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim

        # nonlinearity defaults to 'tanh', can also be 'relu'
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x shape: [Batch, Seq_Len, Input_Dim]
        # Initial hidden state: [Layers, Batch, Hidden_Dim]
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)

        # out shape: [Batch, Seq_Len, Hidden_Dim]
        out, hn = self.rnn(x, h0)

        # Extract the hidden state of the very last time step: out[:, -1, :]
        out = self.fc(out[:, -1, :])
        return out


# =====================================================================
# 2. Long Short-Term Memory Network (LSTM)
# =====================================================================
class LSTMClassifier(nn.Module):
    """
    LSTM Classifier.
    Uses Input, Forget, and Output gates along with a Cell State (c)
    to mitigate vanishing gradients and track long-term dependencies.
    """

    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim

        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # LSTMs maintain two tracking tensors: Hidden state (h0) and Cell state (c0)
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)

        # Forward propagate through sequence
        out, (hn, cn) = self.lstm(x, (h0, c0))

        # Decode the hidden state of the final sequence time-step
        out = self.fc(out[:, -1, :])
        return out


# =====================================================================
# 3. Gated Recurrent Unit Network (GRU)
# =====================================================================
class GRUClassifier(nn.Module):
    """
    GRU Classifier.
    A streamlined LSTM alternative fusing the cell and hidden states.
    Relies on Update and Reset gates; structurally faster to train.
    """

    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim

        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # GRUs simplify back to a single hidden state tracking tensor (h0)
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)

        out, hn = self.gru(x, h0)

        # Decode final step representation
        out = self.fc(out[:, -1, :])
        return out


In [2]:
if __name__ == "__main__":
    # Execution Hyperparameters
    BATCH_SIZE = 16
    SEQ_LENGTH = 25  # Number of tokens/time-steps per sequence
    INPUT_DIM = 64  # Feature vector size per token (e.g., embedding size)
    HIDDEN_DIM = 128  # Recurrent cell state size
    LAYER_DIM = 2  # Stacked/Deep sequential layers
    OUTPUT_DIM = 5  # Target class output dimensions

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running sequential verification loops on: {device}\n")

    # Generate mock sequence data tensor: [Batch, Seq_Len, Features]
    mock_sequence_batch = torch.randn(BATCH_SIZE, SEQ_LENGTH, INPUT_DIM).to(device)
    print(f"Input Shape: {mock_sequence_batch.shape}\n" + "-" * 40)

    # Dictionary mapping for cleaner verification iteration
    models = {
        "Vanilla RNN": RNNClassifier(INPUT_DIM, HIDDEN_DIM, LAYER_DIM, OUTPUT_DIM).to(
            device
        ),
        "LSTM Model": LSTMClassifier(INPUT_DIM, HIDDEN_DIM, LAYER_DIM, OUTPUT_DIM).to(
            device
        ),
        "GRU Model": GRUClassifier(INPUT_DIM, HIDDEN_DIM, LAYER_DIM, OUTPUT_DIM).to(
            device
        ),
    }

    for name, model in models.items():
        # Put models in eval mode for shape profiling
        model.eval()
        with torch.no_grad():
            output = model(mock_sequence_batch)

        print(f"🤖 {name:12} -> Output Matrix Shape: {list(output.shape)}")
        # Check against intended size configuration
        assert output.shape == (BATCH_SIZE, OUTPUT_DIM), (
            "Shape dimension conflict detected!"
        )

    print("-" * 40 + "\nAll sequential tracking verification steps passed cleanly!")


Running sequential verification loops on: cpu

Input Shape: torch.Size([16, 25, 64])
----------------------------------------
🤖 Vanilla RNN  -> Output Matrix Shape: [16, 5]
🤖 LSTM Model   -> Output Matrix Shape: [16, 5]
🤖 GRU Model    -> Output Matrix Shape: [16, 5]
----------------------------------------
All sequential tracking verification steps passed cleanly!
